In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

In [ ]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Augmented (Secondly) Dataset.csv')

# Drop diseases with less than 500 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 500].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 201
Number of rows left: 168499


In [5]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune for KNN
    n_neighbors = trial.suggest_int('n_neighbors', 3, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    p = trial.suggest_int('p', 1, 2)  # p=1 (Manhattan), p=2 (Euclidean)

    # Create KNeighborsClassifier with hyperparameters
    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        p=p
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_neighbors={n_neighbors}, weights={weights}, p={p}, Accuracy={accuracy:.4f}")

    return accuracy

In [7]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize', 
    study_name="knn_diseases_symptoms_dropextremelymore500withoutSMOTE_SecondAugmentation_study",
    storage=r"sqlite:///C:/Users/khiew/Downloads/knn.db", 
    load_if_exists=True
)

# Optimize the study with your objective function, adjust n_trials as needed
study.optimize(objective, n_trials=20)

# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

# After finding the best hyperparameters, you can fit the final KNN model on resampled data
best_params = study.best_trial.params
final_model = KNeighborsClassifier(
    n_neighbors=best_params['n_neighbors'],
    weights=best_params['weights'],
    p=best_params['p']
)
final_model.fit(X_train, y_train)

# Evaluate on test set
y_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")


[I 2025-04-27 13:52:47,541] Using an existing study with name 'knn_diseases_symptoms_dropextremelymore500withoutSMOTE_SecondAugmentation_study' instead of creating a new one.
[I 2025-04-27 13:56:16,748] Trial 4 finished with value: 0.5869405481430588 and parameters: {'n_neighbors': 15, 'weights': 'distance', 'p': 2}. Best is trial 4 with value: 0.5869405481430588.


Trial 4: n_neighbors=15, weights=distance, p=2, Accuracy=0.5869


[I 2025-04-27 14:10:39,157] Trial 5 finished with value: 0.5931127292647821 and parameters: {'n_neighbors': 34, 'weights': 'distance', 'p': 1}. Best is trial 5 with value: 0.5931127292647821.


Trial 5: n_neighbors=34, weights=distance, p=1, Accuracy=0.5931


[I 2025-04-27 14:14:20,848] Trial 6 finished with value: 0.5752861527390257 and parameters: {'n_neighbors': 12, 'weights': 'uniform', 'p': 2}. Best is trial 5 with value: 0.5931127292647821.


...

Trial 21: n_neighbors=38, weights=distance, p=1, Accuracy=0.5935


[I 2025-04-27 17:30:38,374] Trial 22 finished with value: 0.5929569305318341 and parameters: {'n_neighbors': 41, 'weights': 'distance', 'p': 2}. Best is trial 14 with value: 0.5938248885575558.


Trial 22: n_neighbors=41, weights=distance, p=2, Accuracy=0.5930


[I 2025-04-27 17:42:36,063] Trial 23 finished with value: 0.5934836499716076 and parameters: {'n_neighbors': 39, 'weights': 'distance', 'p': 1}. Best is trial 14 with value: 0.5938248885575558.


Trial 23: n_neighbors=39, weights=distance, p=1, Accuracy=0.5935

Best Trial:
FrozenTrial(number=14, state=TrialState.COMPLETE, values=[0.5938248885575558], datetime_start=datetime.datetime(2025, 4, 27, 15, 50, 30, 142045), datetime_complete=datetime.datetime(2025, 4, 27, 16, 4, 18, 668884), params={'n_neighbors': 41, 'weights': 'distance', 'p': 1}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_neighbors': IntDistribution(high=50, log=False, low=3, step=1), 'weights': CategoricalDistribution(choices=('uniform', 'distance')), 'p': IntDistribution(high=2, log=False, low=1, step=1)}, trial_id=182, value=None)
Best Hyperparameters:
{'n_neighbors': 41, 'weights': 'distance', 'p': 1}
Test Accuracy: 0.5930
